# 10kGNAD - A german topic classification dataset.


## 1. imports

## 2. Data

In [ ]:
"""
10kGNAD + Timestamps
====================
Lädt den 10kGNAD-Datensatz herunter und ergänzt ihn mit Publikationsdaten
aus dem One Million Posts Corpus (corpus.sqlite3).

Da 10kGNAD direkt aus dem One Million Posts Corpus extrahiert wurde,
sind ALLE Artikel in beiden Datensätzen identisch vorhanden → 100% Match.
Der Join erfolgt über den Artikeltitel (exakter Prefix-Match).

Output-Spalten:
  label            – Themenklasse (z.B. Wirtschaft, Sport, ...)
  text             – Titel + Artikeltext (wie im Original)
  split            – train / test
  article_id       – ID aus SQLite
  publishing_date  – Publikationsdatum (datetime)
  topic_path       – Voller Pfad (z.B. Newsroom/Wirtschaft/...)
  article_title    – Originaltitel aus SQLite

Voraussetzungen:
  pip install pandas requests tqdm

Lizenzen:
  10kGNAD + One Million Posts Corpus: CC BY-NC-SA 4.0 (nur nicht-kommerziell)
"""

import os
import sqlite3
import tarfile
import requests
import pandas as pd
from tqdm import tqdm


# ── Konfiguration ─────────────────────────────────────────────────────────────

URLS = {
    "train":  "https://raw.githubusercontent.com/tblock/10kGNAD/master/train.csv",
    "test":   "https://raw.githubusercontent.com/tblock/10kGNAD/master/test.csv",
    "corpus": "https://github.com/OFAI/million-post-corpus/releases/download/v1.0.0/million_post_corpus.tar.bz2",
}

DIR            = "../data/10kGNAD"
CORPUS_ARCHIVE = os.path.join(DIR, "million_post_corpus.tar.bz2")
SQLITE_PATH    = os.path.join(DIR, "corpus.sqlite3")
OUTPUT_TRAIN   = os.path.join(DIR, "train_with_timestamps.csv")
OUTPUT_TEST    = os.path.join(DIR, "test_with_timestamps.csv")
OUTPUT_FULL    = os.path.join(DIR, "full_with_timestamps.csv")


# ── Download ──────────────────────────────────────────────────────────────────

def download(url: str, dest: str, label: str = "") -> None:
    """Datei herunterladen mit Fortschrittsbalken. Überspringt bereits vorhandene Dateien."""
    if os.path.exists(dest):
        print(f"  ✓ Vorhanden: {dest}")
        return
    print(f"  ↓ Lade {label or dest} ...")
    r = requests.get(url, stream=True, timeout=300)
    r.raise_for_status()
    total = int(r.headers.get("content-length", 0))
    os.makedirs(os.path.dirname(dest) or ".", exist_ok=True)
    with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True) as bar:
        for chunk in r.iter_content(8192):
            f.write(chunk)
            bar.update(len(chunk))


def extract_sqlite(archive: str, sqlite_dest: str) -> None:
    """SQLite aus tar.bz2-Archiv entpacken."""
    if os.path.exists(sqlite_dest):
        print(f"  ✓ SQLite vorhanden: {sqlite_dest}")
        return
    print("  📦 Entpacke Archiv ...")
    dest_dir = os.path.dirname(sqlite_dest)
    with tarfile.open(archive, "r:bz2") as tar:
        for member in tar.getmembers():
            if member.name.endswith(".sqlite3"):
                member.name = os.path.basename(member.name)
                tar.extract(member, dest_dir)
                print(f"  ✓ Extrahiert: {sqlite_dest}")
                return
        # Fallback: alles entpacken
        tar.extractall(dest_dir)
    # SQLite in entpacktem Verzeichnis suchen
    for root, _, files in os.walk(dest_dir):
        for f in files:
            if f.endswith(".sqlite3"):
                import shutil
                shutil.copy(os.path.join(root, f), sqlite_dest)
                print(f"  ✓ SQLite gefunden: {sqlite_dest}")
                return


# ── Datenladen ────────────────────────────────────────────────────────────────

def load_gnad(split: str) -> pd.DataFrame:
    """10kGNAD CSV laden.
    Format: label;text  (kein Header, Semikolon-getrennt)
    text = Artikeltitel + ' ' + Artikelinhalt (konkateniert vom Extraktionsskript)
    """
    path = os.path.join(DIR, f"{split}.csv")
    df = pd.read_csv(
        path,
        sep=";",
        header=None,
        names=["label", "text"],
        quoting=3,           # QUOTE_NONE – Texte können Anführungszeichen enthalten
        on_bad_lines="skip",
        encoding="utf-8",
    )
    df["split"] = split
    return df


def load_articles(sqlite_path: str) -> pd.DataFrame:
    """Artikel-Metadaten aus dem One Million Posts Corpus laden.

    Die Articles-Tabelle enthält laut Dokumentation:
      ID_Article     INTEGER PRIMARY KEY
      Path           TEXT    Themen-Pfad (Newsroom/Wirtschaft/...)
      publishingDate TEXT    Publikationsdatum
      Title          TEXT
      Body           TEXT
    """
    conn = sqlite3.connect(sqlite_path)

    # Tatsächliche Spaltennamen prüfen
    schema = pd.read_sql("PRAGMA table_info(Articles)", conn)
    print(f"  Articles-Spalten: {list(schema['name'])}")

    articles = pd.read_sql(
        "SELECT ID_Article, publishingDate, Path, Title FROM Articles",
        conn,
    )
    conn.close()
    print(f"  {len(articles):,} Artikel in SQLite geladen")
    return articles


# ── Join ──────────────────────────────────────────────────────────────────────

def add_timestamps(gnad_df: pd.DataFrame, articles_df: pd.DataFrame) -> pd.DataFrame:
    """Timestamps per Titel-Join hinzufügen.

    Das 10kGNAD-Extraktionsskript erzeugt text = Title + ' ' + Body.
    Da alle Artikel aus demselben Corpus stammen, matcht jeder 10kGNAD-Text
    auf genau einen SQLite-Artikel über einen exakten Prefix-Vergleich.
    """
    articles = articles_df.copy()
    articles["title_norm"] = articles["Title"].str.strip().str.lower()

    # Lookup: normalisierter Titel → DataFrame-Index
    lookup = dict(zip(articles["title_norm"], articles.index))

    def find_idx(text: str) -> int | None:
        """Längsten passenden Titel als Prefix suchen."""
        t = str(text).strip().lower()
        best_idx, best_len = None, 0
        for title_norm, idx in lookup.items():
            if t.startswith(title_norm) and len(title_norm) > best_len:
                best_idx = idx
                best_len = len(title_norm)
        return best_idx

    gnad = gnad_df.copy()
    gnad["_idx"] = pd.Series(gnad["text"].apply(find_idx), dtype="Int64")

    matched = gnad["_idx"].notna().sum()
    total   = len(gnad)
    print(f"  Gematcht: {matched:,} / {total:,} ({matched/total*100:.1f}%)")

    if matched < total:
        missing = gnad[gnad["_idx"].isna()]["text"].str[:80].head(5)
        print("  Nicht gematchte Texte (Vorschau):")
        for t in missing:
            print(f"    - {t}")

    # Merge über den gefundenen Index
    meta = articles[["ID_Article", "publishingDate", "Path", "Title"]].rename(columns={
        "ID_Article":     "article_id",
        "publishingDate": "publishing_date",
        "Path":           "topic_path",
        "Title":          "article_title",
    })
    meta.index.name = "_idx"
    meta = meta.reset_index()   # _idx wird reguläre Spalte
    meta["_idx"] = meta["_idx"].astype("Int64")

    result = gnad.merge(meta, on="_idx", how="left").drop(columns=["_idx"])
    result["publishing_date"] = pd.to_datetime(result["publishing_date"], errors="coerce")
    return result


# ── Hauptprogramm ─────────────────────────────────────────────────────────────

def main():
    os.makedirs(DIR, exist_ok=True)

    print("\n=== 1. 10kGNAD herunterladen ===")
    for split in ("train", "test"):
        download(URLS[split], os.path.join(DIR, f"{split}.csv"), f"10kGNAD {split}.csv")

    print("\n=== 2. One Million Posts Corpus herunterladen (~600 MB) ===")
    print("  ⚠  Einmaliger Download, kann einige Minuten dauern.")
    download(URLS["corpus"], CORPUS_ARCHIVE, "One Million Posts Corpus")

    print("\n=== 3. SQLite entpacken ===")
    extract_sqlite(CORPUS_ARCHIVE, SQLITE_PATH)

    print("\n=== 4. Artikel aus SQLite laden ===")
    articles_df = load_articles(SQLITE_PATH)

    print("\n=== 5. 10kGNAD laden ===")
    train_df = load_gnad("train")
    test_df  = load_gnad("test")
    print(f"  Train: {len(train_df):,} | Test: {len(test_df):,}")

    print("\n=== 6. Timestamps hinzufügen ===")
    print("  Train:")
    train_out = add_timestamps(train_df, articles_df)
    print("  Test:")
    test_out  = add_timestamps(test_df, articles_df)
    full_out  = pd.concat([train_out, test_out], ignore_index=True)

    print("\n=== 7. Speichern ===")
    train_out.to_csv(OUTPUT_TRAIN, index=False, encoding="utf-8")
    test_out.to_csv(OUTPUT_TEST,   index=False, encoding="utf-8")
    full_out.to_csv(OUTPUT_FULL,   index=False, encoding="utf-8")
    print(f"  ✓ {OUTPUT_TRAIN}")
    print(f"  ✓ {OUTPUT_TEST}")
    print(f"  ✓ {OUTPUT_FULL}")

    print("\n=== Zusammenfassung ===")
    dates = full_out["publishing_date"].dropna()
    print(f"  Artikel gesamt:  {len(full_out):,}")
    print(f"  Mit Datum:       {len(dates):,} ({len(dates)/len(full_out)*100:.1f}%)")
    if len(dates):
        print(f"  Datum-Bereich:   {dates.min().date()} bis {dates.max().date()}")

    print("\n  Match-Rate pro Klasse:")
    print(
        full_out.groupby("label")["publishing_date"]
        .agg(gesamt="count", mit_datum=lambda x: x.notna().sum())
        .assign(match_pct=lambda df: (df["mit_datum"] / df["gesamt"] * 100).round(1))
        .to_string()
    )

    print("\n  Vorschau:")
    preview = (
        full_out[["label", "split", "publishing_date", "topic_path", "text"]]
        .head(5)
        .assign(text=lambda df: df["text"].str[:60] + "…")
    )
    print(preview.to_string(index=False))
    print("\n✅ Fertig!")


if __name__ == "__main__":
    main()


=== 1. 10kGNAD herunterladen ===
  ✓ Vorhanden: ../data/10kGNAD/train.csv
  ✓ Vorhanden: ../data/10kGNAD/test.csv

=== 2. One Million Posts Corpus herunterladen (~600 MB) ===
  ⚠  Einmaliger Download, kann einige Minuten dauern.
  ✓ Vorhanden: ../data/10kGNAD/million_post_corpus.tar.bz2

=== 3. SQLite entpacken ===
  ✓ SQLite vorhanden: ../data/10kGNAD/corpus.sqlite3

=== 4. Artikel aus SQLite laden ===
  Articles-Spalten: ['ID_Article', 'Path', 'publishingDate', 'Title', 'Body']
  12,087 Artikel in SQLite geladen

=== 5. 10kGNAD laden ===
  Train: 8,620 | Test: 1,009

=== 6. Timestamps hinzufügen ===
  Train:


In [14]:
import pandas as pd

data_dir = "/home/michaelschlee/ownCloud/GIT/LabelFusion/data/10kGNAD"

train_with_timestamps = pd.read_csv(
    f"{data_dir}/train_with_timestamps.csv",
    parse_dates=["publishing_date"],
)

test_with_timestamps = pd.read_csv(
    f"{data_dir}/test_with_timestamps.csv",
    parse_dates=["publishing_date"],
)

print(len(train_with_timestamps), len(test_with_timestamps))


8620 1009


In [15]:
train_with_timestamps.head(2)

,label,text,split,article_id,publishing_date,topic_path,article_title
0,Sport,21-Jähriger fällt wohl bis Saisonende aus. Wie...,train,NaN,NaT,NaN,NaN
1,Web,Der frischgekürte CEO Sundar Pichai setzt auf ...,train,NaN,NaT,NaN,NaN
